In [22]:
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import numpy as np
import semopy
import json
from scipy import stats

In [23]:
colunas_interesse = ["Modelo", "chi2 p-value", "RMSEA", "AIC", "BIC"]
df_modelos_controle_prep = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_controle.csv", index_col="Modelo", usecols=colunas_interesse)
df_modelos_autismo_prep = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_autismo.csv", index_col="Modelo", usecols=colunas_interesse)

df_controle_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle.csv")
df_autismo_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo.csv")

with open(r"..\utils\fm_modelos_5_conexo.json", "r") as file:
    formula_modelos_conexos = json.load(file)

### 1) Filtragem de modelos com base em RMSEA < 0.08

In [24]:
df_controle_filtro1 = df_modelos_controle_prep[df_modelos_controle_prep["RMSEA"] < 0.08]
df_autismo_filtro1 = df_modelos_autismo_prep[df_modelos_autismo_prep["RMSEA"] < 0.08]

print(df_controle_filtro1.shape)
print(df_autismo_filtro1.shape)

(226, 4)
(341, 4)


In [25]:
df_controle_filtro1.sort_values("BIC").head(10)

,chi2 p-value,RMSEA,AIC,BIC
Modelo,,,,
M10302_Edges4,0.999984,0.000000,17.999653,56.438549
M03528_Edges4,1.000000,0.000000,17.999973,56.438869
M00934_Edges4,1.000000,0.000000,17.999999,56.438895
M00001_Edges4,1.000000,0.000000,17.999999,56.438895
M25888_Edges4,1.000000,0.000000,18.000000,56.438896
M09415_Edges5,0.005014,0.066693,19.936700,62.646585
M09551_Edges5,0.005015,0.066693,19.936701,62.646585
M03796_Edges5,0.005489,0.066080,19.937514,62.647398
M01316_Edges5,0.005550,0.066004,19.937614,62.647498


In [26]:
df_autismo_filtro1.sort_values("BIC").head(10)

,chi2 p-value,RMSEA,AIC,BIC
Modelo,,,,
M08682_Edges4,0.034566,0.066740,17.904631,50.777034
M08812_Edges4,0.034628,0.066719,17.904665,50.777067
M25888_Edges4,1.000000,0.000000,17.999999,50.872401
M00001_Edges4,1.000000,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,18.000000,50.872403
M25243_Edges5,0.045759,0.066604,19.920707,56.445599
M25373_Edges5,0.045796,0.066593,19.920722,56.445613


### 2) Selecionando modelos que passaram na etapa 1, simultaneamente para ambos os conjuntos

In [27]:
modelos_filtro2 = pd.merge(df_controle_filtro1, 
                           df_autismo_filtro1, 
                           how="inner", 
                           left_index=True, 
                           right_index=True, 
                           suffixes=["_controle", "_autismo"])
modelos_filtro2.shape

(153, 8)

### 3) Verificar os TOP 10 modelos com base em ordenação de AIC/BIC

#### 3.1) Ordenação com base no AIC/BIC produzido pelo Conjunto Controle:

In [28]:
modelos_filtro2.sort_values("BIC_controle").head(10)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M10302_Edges4,0.999984,0.000000,17.999653,56.438549,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,17.999973,56.438869,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,18.000000,50.872403
M00001_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,17.999999,50.872402
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872401
M09415_Edges5,0.005014,0.066693,19.936700,62.646585,0.179873,0.042767,19.946686,56.471578
M09551_Edges5,0.005015,0.066693,19.936701,62.646585,0.177155,0.043127,19.946378,56.471270
M03796_Edges5,0.005489,0.066080,19.937514,62.647398,0.490646,0.000000,19.968982,56.493874
M01316_Edges5,0.005550,0.066004,19.937614,62.647498,0.490646,0.000000,19.968982,56.493874


Percebe-se que o Top 7 tem uma discrepância em BIC relevante a partir do 8º modelo

In [29]:
modelos_filtro2.sort_values("BIC_controle").head(7)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M10302_Edges4,0.999984,0.000000,17.999653,56.438549,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,17.999973,56.438869,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,18.000000,50.872403
M00001_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,17.999999,50.872402
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872401
M09415_Edges5,0.005014,0.066693,19.936700,62.646585,0.179873,0.042767,19.946686,56.471578
M09551_Edges5,0.005015,0.066693,19.936701,62.646585,0.177155,0.043127,19.946378,56.471270


#### 3.2) Ordenação com base no AIC/BIC produzido pelo Conjunto Autismo:

In [30]:
modelos_filtro2.sort_values("BIC_autismo").head(10)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872401
M00001_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,0.000000,17.999973,56.438869,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,0.999984,0.000000,17.999653,56.438549,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,18.000000,50.872403
M25243_Edges5,0.056869,0.046613,19.959410,62.669294,0.045759,0.066604,19.920707,56.445599
M25373_Edges5,0.047913,0.048386,19.957729,62.667613,0.045796,0.066593,19.920722,56.445613
M00948_Edges5,0.097484,0.040384,19.964819,62.674703,0.131918,0.049470,19.940525,56.465417
M03533_Edges5,0.097491,0.040383,19.964820,62.674704,0.131924,0.049470,19.940526,56.465418


Percebe-se que o Top 7 tem uma discrepância em BIC relevante a partir do 8º modelo

In [31]:
top_7 = modelos_filtro2.sort_values("BIC_autismo").head(7)
top_7_nomes = top_7.index.values
top_7 

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872401
M00001_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,0.000000,17.999973,56.438869,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,0.999984,0.000000,17.999653,56.438549,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999999,56.438895,1.000000,0.000000,18.000000,50.872403
M25243_Edges5,0.056869,0.046613,19.959410,62.669294,0.045759,0.066604,19.920707,56.445599
M25373_Edges5,0.047913,0.048386,19.957729,62.667613,0.045796,0.066593,19.920722,56.445613


### 4) Filtragem por chi quadrado > 0.05

In [32]:
top_filtro_chi = top_7[(top_7["chi2 p-value_autismo"] > 0.05)&(top_7["chi2 p-value_controle"] > 0.05)]
top_filtro_chi

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M25888_Edges4,1.000000,0.0,18.000000,56.438896,1.0,0.0,17.999999,50.872401
M00001_Edges4,1.000000,0.0,17.999999,56.438895,1.0,0.0,17.999999,50.872402
M03528_Edges4,1.000000,0.0,17.999973,56.438869,1.0,0.0,18.000000,50.872403
M10302_Edges4,0.999984,0.0,17.999653,56.438549,1.0,0.0,18.000000,50.872403
M00934_Edges4,1.000000,0.0,17.999999,56.438895,1.0,0.0,18.000000,50.872403


### 5) Verificação de arestas estatisticamente significativas

In [33]:
def roda_os_modelos(df, modelos, ajuste_p=None, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        try:
            formula = modelo["formula"]
        except:
            formula = modelo
        
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            edges_inspection = edges_inspection[edges_inspection["op"] == "~"]
            if ajuste_p:
                p_valores = edges_inspection["p-value"]
                rejeitados, p_corrigidos_fdr, _, _ = multipletests(p_valores, alpha=0.05, method=ajuste_p)
                edges_inspection["p-value-adj"] = p_corrigidos_fdr
                edges_inspection["tipo_correcao"] = ajuste_p
                edges_inspection["rejeita"] = rejeitados
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados

In [34]:
modelos_interesse = {m:v for m,v in formula_modelos_conexos.items() if m in top_filtro_chi.index}

#### 4.1) Arestas indicadas pelo Conjunto de Controle

In [35]:
_, dic_arestas_controle = roda_os_modelos(df_controle_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_controle[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.079024,0.026584,2.972606,2.952837e-03,1.181135e-02,bonferroni,True
1,Control,~,Somatomotor,0.112980,0.021073,5.361447,8.255804e-08,3.302322e-07,bonferroni,True
2,Control,~,Default_mode,0.197885,0.018532,10.678129,0.000000e+00,0.000000e+00,bonferroni,True
3,Control,~,Cerebellar,0.132346,0.026295,5.033127,4.825437e-07,1.930175e-06,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.258256,0.041837,6.172947,6.702885e-10,2.681154e-09,bonferroni,True
1,Cerebellar,~,Somatomotor,0.213666,0.033708,6.338768,2.316096e-10,9.264385e-10,bonferroni,True
2,Cerebellar,~,Default_mode,-0.089754,0.032788,-2.737376,6.193147e-03,2.477259e-02,bonferroni,True
3,Cerebellar,~,Control,0.345616,0.068664,5.033460,4.817050e-07,1.926820e-06,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.309406,0.055575,5.567333,2.586677e-08,1.034671e-07,bonferroni,True
1,Default_mode,~,Somatomotor,0.258288,0.044766,5.769717,7.940460e-09,3.176184e-08,bonferroni,True
2,Default_mode,~,Cerebellar,-0.156165,0.057023,-2.738656,6.169088e-03,2.467635e-02,bonferroni,True
3,Default_mode,~,Control,0.897157,0.084158,10.660346,0.000000e+00,0.000000e+00,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.205601,0.052623,3.907075,9.342014e-05,3.736805e-04,bonferroni,True
1,Somatomotor,~,Default_mode,0.230506,0.039447,5.843415,5.114139e-09,2.045656e-08,bonferroni,True
2,Somatomotor,~,Cerebellar,0.330835,0.051656,6.404565,1.507983e-10,6.031931e-10,bonferroni,True
3,Somatomotor,~,Control,0.456545,0.084327,5.413971,6.164222e-08,2.465689e-07,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.133960,0.034614,3.870114,1.087846e-04,4.351382e-04,bonferroni,True
1,Visual,~,Default_mode,0.179621,0.032213,5.576079,2.460009e-08,9.840037e-08,bonferroni,True
2,Visual,~,Cerebellar,0.260296,0.042172,6.172187,6.735168e-10,2.694067e-09,bonferroni,True
3,Visual,~,Control,0.208066,0.069975,2.973422,2.944994e-03,1.177997e-02,bonferroni,True


#### 4.2) Arestas indicadas pelo Conjunto de Autismo

In [36]:
_, dic_arestas_autismo = roda_os_modelos(df_autismo_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_autismo[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.074549,0.034413,2.166317,0.030287,0.121148,bonferroni,False
1,Control,~,Somatomotor,0.118545,0.030132,3.934170,0.000083,0.000334,bonferroni,True
2,Control,~,Default_mode,0.225208,0.025682,8.768958,0.000000,0.000000,bonferroni,True
3,Control,~,Cerebellar,0.149607,0.034185,4.376418,0.000012,0.000048,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.237138,0.056472,4.199208,2.678504e-05,1.071402e-04,bonferroni,True
1,Cerebellar,~,Somatomotor,0.300495,0.048748,6.164293,7.079877e-10,2.831951e-09,bonferroni,True
2,Cerebellar,~,Default_mode,-0.176080,0.047400,-3.714770,2.033884e-04,8.135538e-04,bonferroni,True
3,Cerebellar,~,Control,0.420842,0.096139,4.377412,1.200970e-05,4.803880e-05,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.371190,0.067536,5.496169,3.881315e-08,1.552526e-07,bonferroni,True
1,Default_mode,~,Somatomotor,0.367117,0.059490,6.171087,6.782217e-10,2.712887e-09,bonferroni,True
2,Default_mode,~,Cerebellar,-0.262094,0.070608,-3.711951,2.056675e-04,8.226701e-04,bonferroni,True
3,Default_mode,~,Control,0.942642,0.107593,8.761193,0.000000e+00,0.000000e+00,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.064500,0.066313,0.972666,3.307193e-01,1.000000e+00,bonferroni,False
1,Somatomotor,~,Default_mode,0.321132,0.052030,6.172059,6.740635e-10,2.696254e-09,bonferroni,True
2,Somatomotor,~,Cerebellar,0.391487,0.063510,6.164174,7.085210e-10,2.834084e-09,bonferroni,True
3,Somatomotor,~,Control,0.434466,0.110405,3.935209,8.312427e-05,3.324971e-04,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.051427,0.052713,0.975616,3.292546e-01,1.000000e+00,bonferroni,False
1,Visual,~,Default_mode,0.258264,0.046953,5.500466,3.787885e-08,1.515154e-07,bonferroni,True
2,Visual,~,Cerebellar,0.245927,0.058484,4.205023,2.610558e-05,1.044223e-04,bonferroni,True
3,Visual,~,Control,0.216487,0.100230,2.159909,3.077970e-02,1.231188e-01,bonferroni,False


### 6) Poda de modelos e comparação com modelos não podados

#### 5.1) Poda de arestas

In [37]:
modelos_podados_bonf = {
    "M00001_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~ 0*Visual + Somatomotor + Default_mode + Cerebellar",
    "M00934_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nControl ~~ Control\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control",
    "M03528_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M10302_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado": "Visual ~ Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado_2": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control"
}

modelos_podados = {
    "M00934_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nControl ~~ Control\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control",
    "M03528_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M10302_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
}

In [38]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo_prep, 
                                        modelos_podados,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle_prep, 
                                        modelos_podados,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M00934_Edges4_podado,0.383607,0.011030,15.971835,50.139742,0.061337,0.057072,15.905435,45.125349
M03528_Edges4_podado,0.381573,0.011284,15.971756,50.139663,0.061336,0.057073,15.905435,45.125348
M10302_Edges4_podado,0.038967,0.045871,15.944132,50.112040,0.995677,0.000000,15.993373,45.213287
M25888_Edges4_podado,0.038969,0.045871,15.944133,50.112040,0.995629,0.000000,15.993350,45.213263



M00934_Edges4_podado:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.233555,0.041144,5.676462,0.0
1,Cerebellar,~,Somatomotor,0.193307,0.033086,5.842504,0.0
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.269024,0.063104,4.263192,0.00002



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.180093,0.055659,3.235664,0.001214
1,Cerebellar,~,Somatomotor,0.247272,0.047722,5.1815,0.0
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.267032,0.088887,3.00418,0.002663


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Somatomotor,~,Visual,0.000000,-,-,-
1,Somatomotor,~,Default_mode,0.338958,0.048794,6.946712,0.0
2,Somatomotor,~,Cerebellar,0.408643,0.061099,6.688252,0.0
3,Somatomotor,~,Control,0.449883,0.10943,4.111141,0.000039



M25888_Edges4_podado:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.216528,0.031213,6.937099,0.0
2,Visual,~,Cerebellar,0.313384,0.040459,7.745627,0.0
3,Visual,~,Control,0.276594,0.068646,4.029295,0.000056



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.275644,0.043409,6.349919,0.0
2,Visual,~,Cerebellar,0.266695,0.054356,4.906484,0.000001
3,Visual,~,Control,0.240018,0.097353,2.46543,0.013685


### 7) Teste de diferença de parâmetros das arestas:

#### 7.1) Teste z

In [39]:
# modelos_podados.update({m:v["formula"] for m, v in modelos_interesse.items()})

In [40]:
for nome, formula in modelos_interesse.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula["formula"])
    model_controle.fit(df_controle_prep)
    arestas_controle = model_controle.inspect()
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula["formula"])
    model_autismo.fit(df_autismo_prep)
    arestas_autismo = model_autismo.inspect()
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()

    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # Cálculo do Z e P-valor usando as colunas mescladas
    z_diff = (df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]) / np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2))
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor"] = p_valor
    
    print(nome)
    display(df_comparacao)


M00001_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Control,~,Visual,0.079024,0.026584,2.972606,2.952837e-03,0.074549,0.034413,2.166317,0.030287,0.102890,0.918050
1,Control,~,Somatomotor,0.112980,0.021073,5.361447,8.255804e-08,0.118545,0.030132,3.934170,0.000083,-0.151341,0.879707
2,Control,~,Default_mode,0.197885,0.018532,10.678129,0.000000e+00,0.225208,0.025682,8.768958,0.000000,-0.862758,0.388271
3,Control,~,Cerebellar,0.132346,0.026295,5.033127,4.825437e-07,0.149607,0.034185,4.376418,0.000012,-0.400228,0.688989


M00934_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Cerebellar,~,Visual,0.258256,0.041837,6.172947,6.702885e-10,0.237138,0.056472,4.199208,2.678504e-05,0.300484,0.763808
1,Cerebellar,~,Somatomotor,0.213666,0.033708,6.338768,2.316096e-10,0.300495,0.048748,6.164293,7.079877e-10,-1.465055,0.142906
2,Cerebellar,~,Default_mode,-0.089754,0.032788,-2.737376,6.193147e-03,-0.176080,0.047400,-3.714770,2.033884e-04,1.497784,0.134189
3,Cerebellar,~,Control,0.345616,0.068664,5.033460,4.817050e-07,0.420842,0.096139,4.377412,1.200970e-05,-0.636741,0.524294


M03528_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Default_mode,~,Visual,0.309406,0.055575,5.567333,2.586677e-08,0.371190,0.067536,5.496169,3.881315e-08,-0.706403,0.479937
1,Default_mode,~,Somatomotor,0.258288,0.044766,5.769717,7.940460e-09,0.367117,0.059490,6.171087,6.782217e-10,-1.461744,0.143811
2,Default_mode,~,Cerebellar,-0.156165,0.057023,-2.738656,6.169088e-03,-0.262094,0.070608,-3.711951,2.056675e-04,1.167155,0.243148
3,Default_mode,~,Control,0.897157,0.084158,10.660346,0.000000e+00,0.942642,0.107593,8.761193,0.000000e+00,-0.332985,0.739146


M10302_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Somatomotor,~,Visual,0.205601,0.052623,3.907075,9.342014e-05,0.064500,0.066313,0.972666,3.307193e-01,1.666763,0.095561
1,Somatomotor,~,Default_mode,0.230506,0.039447,5.843415,5.114139e-09,0.321132,0.052030,6.172059,6.740635e-10,-1.387985,0.165142
2,Somatomotor,~,Cerebellar,0.330835,0.051656,6.404565,1.507983e-10,0.391487,0.063510,6.164174,7.085210e-10,-0.740876,0.458769
3,Somatomotor,~,Control,0.456545,0.084327,5.413971,6.164222e-08,0.434466,0.110405,3.935209,8.312427e-05,0.158927,0.873727


M25888_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Visual,~,Somatomotor,0.133960,0.034614,3.870114,1.087846e-04,0.051427,0.052713,0.975616,3.292546e-01,1.308770,0.190612
1,Visual,~,Default_mode,0.179621,0.032213,5.576079,2.460009e-08,0.258264,0.046953,5.500466,3.787885e-08,-1.381140,0.167236
2,Visual,~,Cerebellar,0.260296,0.042172,6.172187,6.735168e-10,0.245927,0.058484,4.205023,2.610558e-05,0.199279,0.842044
3,Visual,~,Control,0.208066,0.069975,2.973422,2.944994e-03,0.216487,0.100230,2.159909,3.077970e-02,-0.068887,0.945080


### 8) Teste de Invariância global (Teste de Chow)

In [ ]:
modelos_estrela = {
    "M00001_Edges4": "Control ~ Visual + Somatomotor + Default_mode + Cerebellar",
    "M00934_Edges4": "Cerebellar ~ Visual + Somatomotor + Default_mode + Control",
    "M03528_Edges4": "Default_mode ~ Visual + Somatomotor + Cerebellar + Control",
    "M10302_Edges4": "Somatomotor ~ Visual + Default_mode + Cerebellar + Control",
    "M25888_Edges4": "Visual ~ Somatomotor + Default_mode + Cerebellar + Control"
    }

# Assumindo que seus dataframes df_controle e df_autismo já estão carregados
df_total = pd.concat([df_controle_prep, df_autismo_prep])

for nome, formula in modelos_estrela.items():
    # 1. Ajustar os modelos via OLS (Mínimos Quadrados Ordinários)
    modelo_total = smf.ols(formula, data=df_total).fit()
    modelo_controle = smf.ols(formula, data=df_controle_prep).fit()
    modelo_autismo = smf.ols(formula, data=df_autismo_prep).fit()

    # 2. Extrair a Soma dos Quadrados dos Resíduos (SSR - a métrica de erro)
    ssr_total = modelo_total.ssr
    ssr_controle = modelo_controle.ssr
    ssr_autismo = modelo_autismo.ssr


    # 3. Definir os parâmetros para o teste
    k = 5 # 4 preditores + 1 intercepto
    N_controle = len(df_controle_prep) # 529
    N_autismo = len(df_autismo_prep)   # 285

    # 4. Calcular a Estatística F (A 'Tensão' matemática real)
    numerador = (ssr_total - (ssr_controle + ssr_autismo)) / k
    denominador = (ssr_controle + ssr_autismo) / (N_controle + N_autismo - 2 * k)
    estatistica_f = numerador / denominador

    # 5. Calcular o P-valor Global
    graus_liberdade_numerador = k # 5 variáveis
    graus_liberdade_denominador = N_controle + N_autismo - 2 * k
    p_valor = stats.f.sf(estatistica_f, graus_liberdade_numerador, graus_liberdade_denominador)

    print(f"TESTANDO O MODELO {nome}")
    print(f"\n\nEstatística F: {estatistica_f:.4f}")
    print(f"P-valor Global: {p_valor:.6e}")
    print("-" * 55)

    if p_valor < 0.05:
        print("VEREDITO: Quebra Estrutural Detectada (Diferença Sistêmica).")
        print("A topologia de predição da Default_mode difere estatisticamente entre Controle e Autismo.")
    else:
        print("VEREDITO: Invariância Confirmada.")
        print("Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.")

TESTANDO O MODELO M00001_Edges4


Estatística F: 0.9185
P-valor Global: 4.682239e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.
TESTANDO O MODELO M00934_Edges4


Estatística F: 1.9363
P-valor Global: 8.605296e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.
TESTANDO O MODELO M03528_Edges4


Estatística F: 1.8650
P-valor Global: 9.809512e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.
TESTANDO O MODELO M10302_Edges4


Estatística F: 0.8359
P-valor Global: 5.243255e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede 